## Executive Summary

**Purpose:** Analyse permanent-magnet brushed DC motors with a lumped-parameter model.

**What it does:** From geometry and material data it estimates the air-gap flux, armature resistance, motor constant K (= K_t = K_e), rotor inertia, the steady-state speed–torque line, losses and an efficiency map.

### Why It Matters
An engineer needs to estimate torque, speed, and efficiency for a DC motor before prototyping.

### Quick Start
**Inputs:** `GeometricParams` (stator, magnet, air gap, rotor and armature-winding dimensions), `MaterialProperties` (copper, steel, magnet, densities, iron-loss coefficients), `OperatingConditions` (terminal voltage, brush drop and brush resistance).

**Outputs:** `LumpedParams` (R_a, K, J, B_g, Φ), speed–torque characteristic, loss breakdown, efficiency map.

### Known limitations (being reworked)
- The magnetic circuit treats steel as linear (constant permeability) and omits the armature teeth.
- `armature_inductance` and `viscous_damping` in `LumpedParams` are placeholder values, not calculated from the design.
- Armature reaction, leakage flux and brush loss are not modelled.

## How It Fits Into the Motor Design Workflow

**Upstream (depends on):** nothing else in the library yet — the module is self-contained.

**Downstream (used by):** design decisions

In [ ]:
#| default_exp motors.dc_motor
#| export
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Optional

import numpy as np

In [ ]:
#| hide
from typing import Optional
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

## DC Motor (Permanent Magnet Brushed)

A lumped-parameter model for brushed PMDC machines covering magnetic circuit analysis, armature resistance, motor constants, rotor inertia, steady-state torque-speed characteristics, and efficiency mapping.

In [ ]:
#| export
@dataclass
class GeometricParams:
    """Geometric dimensions of the motor (SI units).

    Magnets sit on the stator bore, so `rotor_outer_diameter` should equal
    `stator_inner_diameter - 2*magnet_thickness - 2*air_gap_length`.
    """
    stator_outer_diameter: float  # [m]
    stator_inner_diameter: float  # [m] bore of the stator yoke, where the magnets are mounted
    rotor_outer_diameter: float  # [m] armature outer diameter
    rotor_inner_diameter: float  # [m] armature inner diameter (shaft diameter)
    stack_length: float  # [m]
    air_gap_length: float  # [m] mechanical gap between magnet surface and armature
    num_poles: int  # number of poles (not pole pairs)
    num_armature_slots: int
    slot_opening_width: float  # [m]
    magnet_arc_angle_deg: float  # [deg] mechanical arc covered by one magnet
    magnet_thickness: float  # [m] radial
    turns_per_coil: int  # one coil per slot (double-layer), so Z = 2 * turns_per_coil * num_armature_slots
    armature_wire_cross_section_m2: float  # [m²]
    is_lap_winding: bool = True  # simplex lap (a = num_poles parallel paths); False = simplex wave (a = 2)


@dataclass
class MaterialProperties:
    """Material properties for motor components (SI units)."""
    copper_resistivity: float  # [Ω·m]
    steel_permeability: float  # [H/m] absolute permeability μ0·μr, constant (linear steel)
    magnet_remanent_flux_density: float  # [T] B_r
    magnet_recoil_permeability: float  # [H/m] absolute recoil permeability μ0·μ_rec
    steel_density: float  # [kg/m³]
    copper_density: float  # [kg/m³] (not used yet)
    shaft_density: float  # [kg/m³]
    hysteresis_coeff: float  # k_h [W/(m³·Hz·T^α)] in p_h = k_h f B^α V
    steinmetz_exponent: float  # α in p_h = k_h f B^α V
    eddy_current_coeff: float  # k_e [W/(m³·(Hz·T·m)²)] in p_e = k_e (f B d)² V
    lamination_thickness: float  # [m] d


@dataclass
class OperatingConditions:
    """Terminal operating conditions."""
    terminal_voltage: float  # [V]
    brush_contact_drop_v: float = 1.0  # [V] total contact drop of both brushes
    brush_resistance_total: float = 0.05  # [Ω] in series with the armature


@dataclass
class LumpedParams:
    """Calculated lumped-parameter motor model."""
    armature_resistance: float  # [Ω] R_a, winding only (brushes are in `OperatingConditions`)
    armature_inductance: float  # [H] placeholder, not calculated from the design
    motor_constant: float  # [V·s/rad] = [N·m/A], K = K_e = K_t
    rotor_inertia: float  # [kg·m²]
    viscous_damping: float  # [N·m·s/rad] placeholder, not calculated from the design
    air_gap_flux_density: float  # [T] average over the pole face
    flux_per_pole: float  # [Wb]

In [ ]:
# Example motor used in the examples below.
# Illustrative dimensions and material values only — not a real or validated motor.
mu_0 = 4e-7 * np.pi
geom = GeometricParams(
    stator_outer_diameter=80e-3, stator_inner_diameter=60e-3,
    rotor_outer_diameter=49e-3, rotor_inner_diameter=10e-3,   # 60 - 2*5 - 2*0.5 = 49 mm
    stack_length=40e-3, air_gap_length=0.5e-3,
    num_poles=4, num_armature_slots=18, slot_opening_width=2e-3,
    magnet_arc_angle_deg=60, magnet_thickness=5e-3,
    turns_per_coil=20, armature_wire_cross_section_m2=0.5e-6, is_lap_winding=True,
)
mat = MaterialProperties(
    copper_resistivity=1.72e-8, steel_permeability=2000 * mu_0,
    magnet_remanent_flux_density=0.4, magnet_recoil_permeability=1.05 * mu_0,
    steel_density=7650, copper_density=8900, shaft_density=7850,
    hysteresis_coeff=100.0, steinmetz_exponent=2.0,
    eddy_current_coeff=0.3e6, lamination_thickness=0.5e-3,
)
op = OperatingConditions(terminal_voltage=24.0, brush_contact_drop_v=1.0, brush_resistance_total=0.05)
assert np.isclose(geom.rotor_outer_diameter,
                  geom.stator_inner_diameter - 2 * geom.magnet_thickness - 2 * geom.air_gap_length)

### Carter's Coefficient

Armature slotting increases the effective air gap: flux fringes round the slot openings, so the gap is crossed over a smaller area than the full slot pitch.

**Theory:**
$$g_{eff} = k_c \, g, \qquad k_c = \frac{\tau_s}{\tau_s - \gamma \, g} \ge 1, \qquad \gamma = \frac{(w_{so}/g)^2}{5 + w_{so}/g}$$

where $\tau_s$ is the slot pitch and $w_{so}$ the slot opening. Since $\gamma g < w_{so}$, the denominator stays positive whenever $w_{so} < \tau_s$.

In [ ]:
#| export
def calculate_carter_coefficient(g: float, w_so: float, slot_pitch: float) -> float:
    """Carter's coefficient k_c >= 1 for the slotted armature surface.

    The slot openings lengthen the effective air gap to g_eff = k_c * g.
    k_c = 1 for a slotless surface (w_so = 0).

    Args:
        g: Air gap [m]
        w_so: Slot opening width [m]
        slot_pitch: Slot pitch (distance between slots) [m]

    Raises:
        ValueError: if g or slot_pitch is not positive, or w_so is not in [0, slot_pitch).
    """
    if g <= 0 or slot_pitch <= 0:
        raise ValueError(f"g and slot_pitch must be positive, got g = {g}, slot_pitch = {slot_pitch}")
    if not 0 <= w_so < slot_pitch:
        raise ValueError(f"w_so must be in [0, slot_pitch), got w_so = {w_so}, slot_pitch = {slot_pitch}")
    gamma = (w_so / g) ** 2 / (5 + w_so / g)
    return slot_pitch / (slot_pitch - gamma * g)

In [ ]:
# Examples
# g = 0.5 mm, w_so = 2 mm, slot pitch 20 mm: gamma = 16/9, k_c = 20 / (20 - 0.8889) = 1.0465
assert np.isclose(calculate_carter_coefficient(0.5e-3, 2.0e-3, 20e-3), 1.0465, atol=1e-4)
assert calculate_carter_coefficient(0.5e-3, 0.0, 20e-3) == 1.0       # slotless: no correction

Carter coefficient: 0.9556


AssertionError: 

In [ ]:
#| export
def calculate_magnetic_circuit(geom: GeometricParams, mat: MaterialProperties) -> tuple[float, float]:
    """Magnetic circuit analysis via reluctance network."""
    mu_0 = 4 * np.pi * 1e-7
    pole_face_area = (geom.magnet_arc_angle_deg / 360 * np.pi * 
                      geom.rotor_outer_diameter * geom.stack_length)
    slot_pitch = np.pi * geom.rotor_outer_diameter / geom.num_armature_slots
    k_c = calculate_carter_coefficient(geom.air_gap_length, geom.slot_opening_width, slot_pitch)
    R_gap = (geom.air_gap_length * k_c) / (mu_0 * pole_face_area)

    magnet_area = (geom.magnet_arc_angle_deg / 360 * np.pi * 
                   (geom.stator_inner_diameter - geom.magnet_thickness) * geom.stack_length)
    R_magnet = geom.magnet_thickness / (mat.magnet_recoil_permeability * magnet_area)

    yoke_t = (geom.stator_outer_diameter - geom.stator_inner_diameter) / 2 - geom.magnet_thickness
    R_stator = (np.pi * (geom.stator_outer_diameter - yoke_t) / geom.num_poles / 
                (mat.steel_permeability * yoke_t * geom.stack_length))

    core_t = (geom.rotor_outer_diameter - geom.rotor_inner_diameter) / 2
    R_rotor = (np.pi * (geom.rotor_inner_diameter + core_t) / geom.num_poles / 
               (mat.steel_permeability * core_t * geom.stack_length))

    mmf = (mat.magnet_remanent_flux_density / mat.magnet_recoil_permeability) * geom.magnet_thickness
    R_total = R_magnet + 2 * R_gap + R_stator + R_rotor
    flux_per_pole = mmf / R_total
    B_gap = flux_per_pole / pole_face_area
    return B_gap, flux_per_pole

In [ ]:
# Examples
B_g, flux = calculate_magnetic_circuit(geom, mat)
pole_face_area = geom.magnet_arc_angle_deg / 360 * np.pi * geom.rotor_outer_diameter * geom.stack_length
assert np.isclose(B_g, flux / pole_face_area)
assert 0 < B_g < mat.magnet_remanent_flux_density     # the gap flux density stays below B_r

In [ ]:
#| export
def calculate_armature_resistance(geom: GeometricParams, mat: MaterialProperties) -> float:
    """Armature winding resistance."""
    end_winding = np.pi * (geom.rotor_outer_diameter / 2) / geom.num_poles
    mlt = 2 * geom.stack_length + end_winding
    total_conductors = 2 * geom.turns_per_coil * geom.num_armature_slots
    total_length = (total_conductors / 2) * mlt
    R_wire = mat.copper_resistivity * total_length / geom.armature_wire_cross_section_m2
    parallel_paths = geom.num_poles if geom.is_lap_winding else 2
    return R_wire / parallel_paths ** 2

In [ ]:
# Examples
from dataclasses import replace
# R ∝ 1/a²: a 4-pole lap winding (a = 4) has 1/4 the resistance of the same wire as a wave winding (a = 2)
R_lap = calculate_armature_resistance(geom, mat)
R_wave = calculate_armature_resistance(replace(geom, is_lap_winding=False), mat)
assert np.isclose(R_lap / R_wave, (2 / 4) ** 2)
# on 2 poles lap and wave both have a = 2
g2 = replace(geom, num_poles=2)
assert np.isclose(calculate_armature_resistance(g2, mat),
                  calculate_armature_resistance(replace(g2, is_lap_winding=False), mat))

In [ ]:
#| export
def calculate_motor_constants(geom: GeometricParams, flux_per_pole: float) -> float:
    """Motor constant K = Kt = Ke."""
    total_conductors = 2 * geom.turns_per_coil * geom.num_armature_slots
    parallel_paths = geom.num_poles if geom.is_lap_winding else 2
    return (total_conductors * geom.num_poles * flux_per_pole) / (2 * np.pi * parallel_paths)

In [ ]:
# Examples
# K = Z P Φ / (2π a): Z = 2*20*18 = 720 conductors, P = 4, a = 4  ->  K = 720 Φ / (2π)
assert np.isclose(calculate_motor_constants(geom, 1e-3), 720 * 1e-3 / (2 * np.pi))
# a wave winding (a = 2) on 4 poles doubles K for the same conductors
assert np.isclose(calculate_motor_constants(replace(geom, is_lap_winding=False), 1e-3),
                  2 * calculate_motor_constants(geom, 1e-3))

In [ ]:
#| export
def calculate_rotor_inertia(geom: GeometricParams, mat: MaterialProperties) -> float:
    """Rotor moment of inertia."""
    r_o = geom.rotor_outer_diameter / 2
    r_i = geom.rotor_inner_diameter / 2
    vol_core = np.pi * (r_o ** 2 - r_i ** 2) * geom.stack_length
    J_core = 0.5 * mat.steel_density * vol_core * (r_o ** 2 + r_i ** 2)
    J_winding = J_core * 0.15
    vol_shaft = np.pi * r_i ** 2 * geom.stack_length
    J_shaft = 0.5 * mat.shaft_density * vol_shaft * r_i ** 2
    return J_core + J_winding + J_shaft

In [ ]:
# Examples
r_o, r_i, L = geom.rotor_outer_diameter / 2, geom.rotor_inner_diameter / 2, geom.stack_length
J_core = 0.5 * mat.steel_density * np.pi * (r_o**2 - r_i**2) * L * (r_o**2 + r_i**2)   # hollow cylinder
J_shaft = 0.5 * mat.shaft_density * np.pi * r_i**2 * L * r_i**2                       # solid cylinder
assert np.isclose(calculate_rotor_inertia(geom, mat), 1.15 * J_core + J_shaft)       # +15 % for the winding

In [ ]:
#| export
def calculate_lumped_parameters(geom: GeometricParams, mat: MaterialProperties) -> LumpedParams:
    """Complete lumped model."""
    B_g, flux_per_pole = calculate_magnetic_circuit(geom, mat)
    Ra = calculate_armature_resistance(geom, mat)
    K = calculate_motor_constants(geom, flux_per_pole)
    J = calculate_rotor_inertia(geom, mat)
    La = 10 * Ra * 1e-6
    b = J * 0.01
    return LumpedParams(
        armature_resistance=Ra, armature_inductance=La, motor_constant=K,
        rotor_inertia=J, viscous_damping=b,
        air_gap_flux_density=B_g, flux_per_pole=flux_per_pole,
    )

In [ ]:
# Examples
params = calculate_lumped_parameters(geom, mat)
assert params.armature_resistance == calculate_armature_resistance(geom, mat)
assert params.motor_constant == calculate_motor_constants(geom, params.flux_per_pole)
assert params.rotor_inertia == calculate_rotor_inertia(geom, mat)

In [ ]:
#| export
def calculate_steady_state_performance(params: LumpedParams, op: OperatingConditions) -> dict:
    """Steady-state speed-torque characteristic.

    The circuit resistance is R_a plus `op.brush_resistance_total`; the brush
    contact drop is subtracted from the terminal voltage.
    """
    K = params.motor_constant
    R = params.armature_resistance + op.brush_resistance_total
    V_eff = max(0.0, op.terminal_voltage - op.brush_contact_drop_v)
    stall_torque = (V_eff * K) / R
    torque_range = np.linspace(0, stall_torque, 200)
    speed_rad_s = V_eff / K - (R / K ** 2) * torque_range
    speed_rpm = speed_rad_s * 60 / (2 * np.pi)
    return {
        "torque": torque_range, "speed_rpm": speed_rpm,
        "current": torque_range / K,
        "no_load_speed_rpm": (V_eff / K) * 60 / (2 * np.pi),
        "stall_torque": stall_torque, "effective_voltage": V_eff,
    }

In [ ]:
# Examples
ss = calculate_steady_state_performance(params, op)
K, R = params.motor_constant, params.armature_resistance + op.brush_resistance_total
V_eff = op.terminal_voltage - op.brush_contact_drop_v
assert np.isclose(ss["no_load_speed_rpm"], V_eff / K * 60 / (2 * np.pi))
assert np.isclose(ss["stall_torque"], K * V_eff / R)     # brush resistance in series with R_a
assert np.isclose(ss["speed_rpm"][-1], 0, atol=1e-6)      # zero speed at stall

In [ ]:
#| export
def calculate_losses_and_efficiency(
    params: LumpedParams, geom: GeometricParams, mat: MaterialProperties,
    torque: float, speed_rpm: float,
) -> dict:
    """Loss breakdown and efficiency."""
    if speed_rpm < 0 or torque < 0:
        return {"p_cu": 0, "p_iron": 0, "p_mech": 0, "p_out": 0, "efficiency": 0}
    omega = speed_rpm * 2 * np.pi / 60
    current = torque / params.motor_constant if params.motor_constant > 0 else 0
    p_cu = current ** 2 * params.armature_resistance
    p_mech = params.viscous_damping * omega ** 2
    f_e = (geom.num_poles / 2) * (omega / (2 * np.pi))
    vol_rotor = (np.pi * ((geom.rotor_outer_diameter / 2) ** 2 - 
                          (geom.rotor_inner_diameter / 2) ** 2) * geom.stack_length)
    B = params.air_gap_flux_density
    p_hyst = mat.hysteresis_coeff * f_e * B ** mat.steinmetz_exponent * vol_rotor
    p_eddy = mat.eddy_current_coeff * (f_e * B * mat.lamination_thickness) ** 2 * vol_rotor
    p_iron = p_hyst + p_eddy
    p_out = torque * omega
    p_in = p_out + p_cu + p_mech + p_iron
    eff = p_out / p_in if p_in > 0 else 0.0
    return {"p_cu": p_cu, "p_iron": p_iron, "p_mech": p_mech, "p_out": p_out, "efficiency": eff}

In [ ]:
# Examples
T, n = 0.5 * ss["stall_torque"], 0.5 * ss["no_load_speed_rpm"]
d = calculate_losses_and_efficiency(params, geom, mat, T, n)
assert np.isclose(d["p_out"], T * n * 2 * np.pi / 60)
assert 0 < d["efficiency"] < 1

In [ ]:
#| export
def plot_efficiency_map(
    params: LumpedParams, geom: GeometricParams, mat: MaterialProperties,
    op_conditions: OperatingConditions, steady_state_data: dict,
    target_torque: Optional[float] = None, target_speed_rpm: Optional[float] = None,
):
    """2-D efficiency contour map."""
    try:
        import matplotlib.pyplot as plt
    except ImportError as e:
        raise ImportError("matplotlib required") from e
    torque_max = steady_state_data["stall_torque"]
    speed_max = steady_state_data["no_load_speed_rpm"]
    V_eff = steady_state_data.get("effective_voltage",
            op_conditions.terminal_voltage - op_conditions.brush_contact_drop_v)
    torque_grid = np.linspace(0.01, torque_max, 50)
    speed_grid = np.linspace(0.01, speed_max, 50)
    T, S = np.meshgrid(torque_grid, speed_grid)
    Eff = np.zeros_like(T)
    K = params.motor_constant
    R = params.armature_resistance + op_conditions.brush_resistance_total
    for i in range(T.shape[0]):
        for j in range(T.shape[1]):
            max_t = (K ** 2 / R) * (V_eff / K - S[i, j] * 2 * np.pi / 60)
            if T[i, j] <= max_t and max_t > 0:
                d = calculate_losses_and_efficiency(params, geom, mat, T[i, j], S[i, j])
                Eff[i, j] = d["efficiency"]
            else:
                Eff[i, j] = np.nan
    fig, ax = plt.subplots(figsize=(10, 6))
    cf = ax.contourf(T, S, Eff * 100, levels=np.arange(0, 101, 5), cmap="viridis")
    fig.colorbar(cf, ax=ax, label="Efficiency (%)")
    ax.plot(steady_state_data["torque"], steady_state_data["speed_rpm"],
            "r--", linewidth=2, label="Operating Envelope")
    if target_torque is not None and target_speed_rpm is not None:
        max_t_at_target = (K ** 2 / R) * (V_eff / K - target_speed_rpm * 2 * np.pi / 60)
        in_env = target_torque <= max_t_at_target and max_t_at_target > 0
        color = "lime" if in_env else "red"
        ax.plot(target_torque, target_speed_rpm, "*", color=color, markersize=16)
    ax.set_xlabel("Torque (Nm)")
    ax.set_ylabel("Speed (RPM)")
    ax.set_title("Motor Efficiency Map")
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.legend()
    fig.tight_layout()
    return fig

In [ ]:
# Examples
fig = plot_efficiency_map(params, geom, mat, op, ss)
assert fig.axes[0].get_xlabel() == "Torque (Nm)"
plt.close(fig)

## Tests

In [ ]:
# Test Carter coefficient: k_c >= 1 and it grows with the slot opening
k_small = calculate_carter_coefficient(1e-3, 0.5e-3, 5e-3)
k_large = calculate_carter_coefficient(1e-3, 2.0e-3, 5e-3)
assert 1 < k_small < k_large
for bad in [(0, 1e-3, 5e-3), (1e-3, 1e-3, 0), (1e-3, 5e-3, 5e-3), (1e-3, -1e-3, 5e-3)]:
    try:
        calculate_carter_coefficient(*bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"no ValueError for {bad}")
print(f"✓ Carter coefficient: {k_small:.4f} < {k_large:.4f}")

# Test brush resistance lowers stall torque, but not no-load speed
ss0 = calculate_steady_state_performance(params, replace(op, brush_resistance_total=0.0))
ss1 = calculate_steady_state_performance(params, replace(op, brush_resistance_total=0.5))
assert ss1["stall_torque"] < ss0["stall_torque"]
assert np.isclose(ss1["no_load_speed_rpm"], ss0["no_load_speed_rpm"])
print("✓ Brush resistance in series with the armature")

✓ Carter coefficient: 0.9909


## References

- Carter, F.W. (1901). Air-gap induction. *Electrical World and Engineer*, 38, 884–888.
- Pyrhönen, J., Jokinen, T., Hrabovcová, V. (2008). *Design of Rotating Electrical Machines*. Wiley — Carter's coefficient (γ form).
- Chapman, S.J. (2005). *Electric Machinery Fundamentals*. McGraw-Hill — DC machine EMF and torque, K = ZPΦ/(2πa).
- Fitzgerald, A.E., Kingsley, C., Umans, S.D. (2003). *Electric Machinery*. McGraw-Hill.